# Chebyshev近似でWigner dを計算する

このnotebookでは、次の2点だけを確認します。

1. GPU版とCPU版の2つの経路でWigner dのChebyshevテーブルを作る
2. 1つの `(l, M, n, beta)` について、両テーブルの値を直接GPU計算と比較する

In [ ]:
using Pkg

function find_repo_root(start=pwd())
    current = abspath(start)
    while true
        if isfile(joinpath(current, "Project.toml")) &&
            isfile(joinpath(current, "src", "EBCoDeCMB.jl"))
            return current
        end
        parent = dirname(current)
        parent == current && error("EBCoDeCMB.jl repository root was not found")
        current = parent
    end
end

repo_root = find_repo_root()
Pkg.activate(repo_root; io=devnull)

using CUDA
using EBCoDeCMB

CUDA.functional() || error("A functional CUDA GPU is required")
(device=string(CUDA.device()), project=basename(repo_root))

## 1. GPU版とCPU版の2経路でテーブルを作る

両方とも同じ `cp` と `scan_plan` を受け取り、同じ構造の `WignerChebyshevTable` を返します。GPU版は係数をGPUで生成し、CPU版は確認用の独立したCPU recurrenceで生成します。

In [ ]:
cp = ConvolutionParams(
    nside_output=8,
    lstart=1,
    lstop=16,
    lmax=16,
    bnmax=2,
    κ=2,
)

scan_plan = ScanMomentPlan(
    kappa=cp.κ,
    nmax=calc_nmax(cp, cp.lstop),
    beta_order=16,
    beta_max=0.2,
    precision=Float64,
)

# GPU経路
table_gpu = CUDA.@sync build_wigner_chebyshev_table_gpu(cp, scan_plan)

# CPU経路（GPUがなくても実行可能）
table_cpu = build_wigner_chebyshev_table_cpu(cp, scan_plan)

(
    gpu_shape=size(table_gpu.coefficients),
    cpu_shape=size(table_cpu.coefficients),
    pair_count=length(table_gpu.pair_M),
    same_layout=(
        table_gpu.pair_M == table_cpu.pair_M &&
        table_gpu.pair_n == table_cpu.pair_n
    ),
)

## 2. 1ケースだけ精度を比較する

`(l, M, n, beta) = (8, 2, 1, 0.1)` を使います。同じChebyshev基底を使って、GPU版とCPU版それぞれの係数からWigner dを再構成します。

In [ ]:
l, M, n = 8, 2, 1
beta = 0.10

pair_index = findfirst(eachindex(table_gpu.pair_M)) do index
    table_gpu.pair_M[index] == M && table_gpu.pair_n[index] == n
end
isnothing(pair_index) && error("(M, n) pair was not found in the table")
l_index = l - table_gpu.lstart + 1

basis = zeros(eltype(table_gpu.coefficients), scan_plan.beta_order)
EBCoDeCMB._chebyshev_values!(basis, scan_plan, beta)
d_gpu = sum(
    @view(table_gpu.coefficients[:, pair_index, l_index]) .* basis,
)
d_cpu = sum(
    @view(table_cpu.coefficients[:, pair_index, l_index]) .* basis,
)

(l=l, M=M, n=n, beta=beta, gpu=d_gpu, cpu=d_cpu)

最後に、同じ1ケースを `ringd_blocks_recursion_gpu` で直接計算し、GPU版・CPU版それぞれの絶対誤差を確認します。

In [ ]:
ringd_device = CUDA.@sync ringd_blocks_recursion_gpu(
    cp,
    beta;
    precision=Float64,
)
ringd = Array(ringd_device)
CUDA.unsafe_free!(ringd_device)

Mmax_l = min(calc_Mmax(cp, l), l)
d_direct = real(ringd[M + l + 1, n + Mmax_l + 1, l_index])
gpu_absolute_error = abs(d_gpu - d_direct)
cpu_absolute_error = abs(d_cpu - d_direct)
cpu_gpu_difference = abs(d_cpu - d_gpu)

@assert max(gpu_absolute_error, cpu_absolute_error) < 1e-8
(
    case=(l=l, M=M, n=n, beta=beta),
    gpu_chebyshev=d_gpu,
    cpu_chebyshev=d_cpu,
    direct=d_direct,
    gpu_absolute_error=gpu_absolute_error,
    cpu_absolute_error=cpu_absolute_error,
    cpu_gpu_difference=cpu_gpu_difference,
)